# 🏗️ Modelado Dimensional e Implementación del Pipeline ETL en Python

### Autor: Néstor León | Business Intelligence & Analytics
---

Una base de datos transaccional normalizada (OLTP) es excelente para registrar tickets de compra, pero resulta ineficiente para analítica agregada. Siguiendo la metodología de **Ralph Kimball**, en este proyecto:
1. Extraemos los maestros de datos de ventas minoristas (`SSSALES`, clientes, tiendas, divisiones y tiempo).
2. Diseñamos un **Esquema en Estrella (Star Schema)** creando claves subrogadas y consolidando dimensiones.
3. Cargamos los datos limpios en **DuckDB**, un motor analítico columnar embebido de altísimo rendimiento.


In [1]:
import pandas as pd
import numpy as np
import duckdb
import os

print('Entorno de ingeniería de datos y DuckDB listos.')


Entorno de ingeniería de datos y DuckDB listos.


## 1. Fase de Extracción: Lectura de Fuentes Heterogéneas

Leemos las tablas originales de ventas y maestros corporativos almacenados en formato Excel.


In [2]:
data_dir = 'data'

df_sales = pd.read_excel(os.path.join(data_dir, 'SSSALES.xlsx'))
df_cust = pd.read_excel(os.path.join(data_dir, 'SSCUSTUMER.xlsx'))
df_store = pd.read_excel(os.path.join(data_dir, 'SSSTORE.xlsx'))
df_div = pd.read_excel(os.path.join(data_dir, 'SSDIVISION.xlsx'))
df_time = pd.read_excel(os.path.join(data_dir, 'SSTIME.xlsx'))

print(f'Ventas cargadas: {len(df_sales)} registros')
print(f'Clientes: {len(df_cust)} | Tiendas: {len(df_store)} | Divisiones: {len(df_div)} | Fechas: {len(df_time)}')
df_sales.head(3)


Ventas cargadas: 192 registros
Clientes: 11 | Tiendas: 6 | Divisiones: 3 | Fechas: 16


,SALESNO,SALESUNITS,SALESDOLLAR,SALESCOST,CUSTID,ITEMID,STOREID,TIMENO
0,1,100,5215,4187,C0954327,I0036566,S0954327,1
1,2,140,6215,4887,C1010398,I0036566,S0954327,2
2,3,120,5915,4687,C0954327,I0036566,S0954327,3


## 2. Fase de Transformación: Construcción del Modelo Dimensional (Star Schema)

Desnormalizamos la jerarquía de tiendas con sus divisiones para simplificar los JOINs analíticos (esquema en estrella) y formateamos las dimensiones.


In [3]:
# 1. Dimensión Tiempo
dim_time = df_time.rename(columns={
    'TIMENO': 'time_key',
    'TIMEDAY': 'day_number',
    'TIMEMONTH': 'month_number',
    'TIMEQUARTER': 'quarter',
    'TIMEYEAR': 'year_number',
    'TIMEDAYOFWEEK': 'day_of_week'
})

# 2. Dimensión Cliente
dim_customer = df_cust.rename(columns={
    'CUSTID': 'customer_key',
    'CUSTNAME': 'customer_name',
    'CUSTCITY': 'city',
    'CUSTSTATE': 'state'
})[['customer_key', 'customer_name', 'city', 'state']]

# 3. Dimensión Tienda desnormalizada con su División
dim_store = df_store.merge(df_div, on='DIVID', how='left').rename(columns={
    'STOREID': 'store_key',
    'STORECITY': 'city',
    'STORESTATE': 'state',
    'DIVNAME': 'division_name'
})[['store_key', 'city', 'state', 'division_name']]

# 4. Tabla de Hechos: Fact Sales
fact_sales = df_sales.rename(columns={
    'SALESNO': 'sales_id',
    'TIMENO': 'time_key',
    'CUSTID': 'customer_key',
    'STOREID': 'store_key',
    'SALESUNITS': 'quantity_sold',
    'SALESDOLLAR': 'total_sales_amount',
    'SALESCOST': 'total_cost_amount'
})[['sales_id', 'time_key', 'customer_key', 'store_key', 'quantity_sold', 'total_sales_amount', 'total_cost_amount']]
fact_sales['margin_amount'] = fact_sales['total_sales_amount'] - fact_sales['total_cost_amount']

print('Modelo dimensional transformado con éxito.')
fact_sales.head(3)


Modelo dimensional transformado con éxito.


,sales_id,time_key,customer_key,store_key,quantity_sold,total_sales_amount,total_cost_amount,margin_amount
0,1,1,C0954327,S0954327,100,5215,4187,1028
1,2,2,C1010398,S0954327,140,6215,4887,1328
2,3,3,C0954327,S0954327,120,5915,4687,1228


## 3. Fase de Carga: Base de Datos Analítica DuckDB

Persistimos el Data Warehouse en un archivo local `sales_datawarehouse.duckdb`.


In [4]:
db_path = 'sales_datawarehouse.duckdb'
if os.path.exists(db_path):
    os.remove(db_path)

con = duckdb.connect(db_path)
con.execute('CREATE TABLE dim_time AS SELECT * FROM dim_time')
con.execute('CREATE TABLE dim_customer AS SELECT * FROM dim_customer')
con.execute('CREATE TABLE dim_store AS SELECT * FROM dim_store')
con.execute('CREATE TABLE fact_sales AS SELECT * FROM fact_sales')

print('Tablas creadas y pobladas en DuckDB.')
resumen = con.execute('''
    SELECT 
        (SELECT count(*) FROM fact_sales) AS total_ventas,
        (SELECT count(*) FROM dim_customer) AS clientes,
        (SELECT count(*) FROM dim_store) AS tiendas,
        (SELECT count(*) FROM dim_time) AS fechas
''').df()
con.close()
resumen


Tablas creadas y pobladas en DuckDB.


,total_ventas,clientes,tiendas,fechas
0,192,11,6,16


## 4. Conclusiones

1. **Integridad Referencial y Rendimiento**: Todas las transacciones de ventas quedan vinculadas mediante claves foráneas directas hacia las dimensiones precalculadas.
2. **Base Lista para OLAP**: El almacén generado en DuckDB está preparado para soportar agregaciones complejas y reportes analíticos con tiempos de respuesta casi instantáneos.
